In [0]:
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

In [0]:
# read the products data from bronze table

product_df = (
    spark.read.table("retail_lakehouse.bronze.products")
)

display(product_df)

In [0]:
# clean product rules standarizes
from pyspark.sql import functions as F

product_validation_df = (
    product_df
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn("active_flag", F.trim(F.col("active_flag")))

    .withColumn(
        "updated_at",
        F.to_timestamp(
            F.col("updated_at"),
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    .withColumn(
        "dq_product_id",
        F.col("product_id").isNotNull()
    )

    .withColumn(
        "dq_product_name",
        F.col("product_name").isNotNull()
    )

    .withColumn(
        "dq_category",
        F.col("category").isNotNull()
    )

    .withColumn(
        "dq_unit_price",
        F.col("unit_price") > 0
    )

    .withColumn(
        "dq_active_flag",
        F.col("active_flag").isin("Y", "N")
    )

    .withColumn(
        "dq_updated_at",
        F.col("updated_at").isNotNull()
    )

    .withColumn(
        "dq_product",
        F.col("updated_at").isNotNull()
    )
)

In [0]:
display(
    product_validation_df.select(
        "product_id",
        "product_name",
        "category",
        "unit_price",
        "active_flag",
        "updated_at",
        "dq_product_id",
        "dq_product_name",
        "dq_category",
        "dq_unit_price",
        "dq_active_flag",
        "dq_updated_at",
    )
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

product_window = Window.partitionBy("product_id")

product_validation_df = (
    product_validation_df

    # Product ID must not be null
    .withColumn(
        "dq_product_id_not_null",
        F.col("product_id").isNotNull()
    )

    # Count how many times each product_id occurs
    .withColumn(
        "product_id_count",
        F.count("*").over(product_window)
    )

    # Product ID must be unique
    .withColumn(
        "dq_product_id_unique",
        F.col("product_id_count") == 1
    )
)

display(
    product_validation_df.select(
        "product_id",
        "product_id_count",
        "dq_product_id_not_null",
        "dq_product_id_unique"
    )
)

In [0]:
# combine all rules 

product_validation_df = (
    product_validation_df
    .withColumn(
        "dq_status",
        F.when(
            F.col("dq_product_id_not_null") &
            F.col("dq_product_id_unique") &
            F.col("dq_product_name") &
            F.col("dq_category") &
            F.col("dq_unit_price") &
            F.col("dq_active_flag") &
            F.col("dq_updated_at"),
            "Pass"
            ).otherwise("Fail"
        )
    )
)

In [0]:
product_validation_df.groupBy("dq_status").count().display()

In [0]:
display(
    product_validation_df
    .filter(F.col("dq_status") == "Fail")
)

In [0]:
# split the valid rows and invalid rows

valid_product_df  = (
    product_validation_df
    .filter(F.col("dq_status") == "Pass")
)

Invalid_product_df = (
    product_validation_df
    .filter(F.col("dq_status") == "Fail")
)

print("Valid:", valid_product_df.count())
print("Invalid:", Invalid_product_df.count())

In [0]:
# create a 2 delta tables in silver layer 
spark.sql("USE CATALOG retail_lakehouse")
spark.sql("USE SCHEMA sliver")

(
    valid_product_df
    .write 
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_lakehouse.sliver.products")

)

(
    Invalid_product_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("retail_lakehouse.sliver.products_quarantine")
)